## Section 1: Setup & Load

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('data/raw')
OUTPUT_DIR = Path('outputs/eda'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv(DATA_DIR / 'store.csv')
train = train.merge(store, on='Store', how='left')
train = train[(train['Open'] == 1) & (train['Sales'] > 0)].copy()

print(f'Train shape: {train.shape}')
print(f"Date range: {train['Date'].min()} -> {train['Date'].max()}")
print(f"Stores: {train['Store'].nunique()}")

## Section 2: Target Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train['Sales'], bins=80, edgecolor='none', alpha=0.8)
axes[0].set_title('Sales distribution (raw)')
axes[0].set_xlabel('Daily sales (EUR)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'EUR {x:,.0f}'))

axes[1].hist(np.log1p(train['Sales']), bins=80, edgecolor='none', alpha=0.8)
axes[1].set_title('Sales distribution (log1p)')
axes[1].set_xlabel('log(1 + sales)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig01_sales_distribution.png', dpi=150)
plt.show()

# FINDING: Raw sales are right-skewed and log1p is more stable for modeling.
print(f"Sales skewness (raw): {train['Sales'].skew():.3f}")
print(f"Sales skewness (log1p): {np.log1p(train['Sales']).skew():.3f}")

## Section 3: Temporal Patterns

In [ ]:
dow_sales = train.groupby('DayOfWeek')['Sales'].agg(['mean','std']).reset_index()
dow_sales['DayOfWeek_name'] = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(dow_sales['DayOfWeek_name'], dow_sales['mean'], yerr=dow_sales['std']/10, capsize=4, alpha=0.85)
ax.set_title('Mean daily sales by day of week')
ax.set_ylabel('Mean sales (EUR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'EUR {x:,.0f}'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig02_dow_seasonality.png', dpi=150)
plt.show()
# FINDING: Day-of-week effect is strong, validating a known-future weekday feature.

In [ ]:
train['month'] = train['Date'].dt.month
monthly = train.groupby('month')['Sales'].mean()
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(monthly.index, monthly.values, marker='o', linewidth=2)
ax.set_title('Mean daily sales by month')
ax.set_xlabel('Month')
ax.set_ylabel('Mean sales (EUR)')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'EUR {x:,.0f}'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig03_monthly_seasonality.png', dpi=150)
plt.show()
# FINDING: Seasonality peaks and troughs support monthly and holiday covariates.

## Section 4: Promo Effects

In [ ]:
promo_effect = train.groupby('Promo')['Sales'].agg(['mean','median','count'])
promo_effect.index = ['No promo', 'Promo']
uplift_mean = (promo_effect.loc['Promo','mean'] / promo_effect.loc['No promo','mean'] - 1) * 100
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['No promo', 'Promo'], promo_effect['mean'], alpha=0.85)
ax.set_title('Mean sales: promo vs no promo')
ax.set_ylabel('Mean sales (EUR)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig05_promo_effect.png', dpi=150)
plt.show()
# FINDING: Promotions create a measurable uplift and should remain in known features.
print(f'Promo mean uplift: +{uplift_mean:.1f}%')

## Section 5: Store Heterogeneity

In [ ]:
store_cv = train.groupby('Store')['Sales'].agg(mean='mean', std='std').assign(cv=lambda df: df['std'] / df['mean'])
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(store_cv['cv'], bins=50, alpha=0.85)
ax.axvline(store_cv['cv'].median(), linestyle='--', label=f"Median CV = {store_cv['cv'].median():.2f}")
ax.set_title('Per-store coefficient of variation')
ax.set_xlabel('CV')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig07_store_cv.png', dpi=150)
plt.show()
# FINDING: CV spread motivates curriculum-style RL and store difficulty grouping.

## Section 6: Summary Statistics for Paper

In [ ]:
summary = {
    'Dataset': 'Rossmann Store Sales',
    'Total records (open, sales>0)': int(len(train)),
    'Number of stores': int(train['Store'].nunique()),
    'Date range': f"{train['Date'].min().date()} to {train['Date'].max().date()}",
    'Mean daily sales': f"EUR {train['Sales'].mean():,.0f}",
    'Std daily sales': f"EUR {train['Sales'].std():,.0f}",
    'Promo coverage': f"{(train['Promo']==1).mean()*100:.1f}%",
    'Holiday coverage': f"{(train['StateHoliday']!='0').mean()*100:.1f}%",
}
for key, value in summary.items():
    print(f'{key:35s}: {value}')
# FINDING: Summary confirms broad store coverage and clear event-driven demand effects.